# MuSiQue · Prolog instead of ONA (ablation, local)

Is it NARS that makes the difference, or would any logic engine do? This notebook takes the saved NARS runs
(`results/traces/<NARS system>/`) and gives **exactly the same triples and rule sets** to SWI-Prolog instead of ONA:
same questions, same k, same facts, same rules, same voting over rule sets. Only the reasoner changes. No new LLM
calls (only the judge grades answers that are not exact, a few cents).

What Prolog does not have, by design: truth values, preferring the answer type the question asks for, partial
matching. It proves a binding or it does not (details at the top of `common/prolog.py`).

**Needs:** SWI-Prolog (`brew install swi-prolog`), and the NARS traces from Colab copied into `results/traces/`.
Answers go to `results/answers/<Prolog system>/k<k>.jsonl`, so the report shows them next to every other system.

In [ ]:
import json, os, subprocess, sys
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
sys.path.insert(0, "../common")
import experiment as ex
import prolog

DATASET = "MuSiQue"
N_QUESTIONS = 200
SOURCE = "NARS + 8B + LoRA"          # whose triples and rules Prolog reuses
SYSTEM = "Prolog + 8B + LoRA"        # the name of this row in the results
K_VALUES = ex.K_VALUES[DATASET]
ex.SIZES.setdefault(SYSTEM, "8B")

print(subprocess.run([prolog.SWIPL, "--version"], capture_output=True, text=True).stdout.strip())
questions, question_hash = ex.load_questions("questions.pkl", N_QUESTIONS)
ex.guard("results", question_hash)
traces = {k: prolog.load_traces("results", SOURCE, k) for k in K_VALUES}
for k in K_VALUES:
    print(f"k={k:<2}: {len(traces[k])} {SOURCE} traces")
assert any(traces.values()), f"no traces for {SOURCE} in results/traces/: copy them from Colab first"

In [ ]:
judge = ex.deepinfra()


def run_one(index, k):
    q, trace = questions[index], traces[k][index]
    answer, votes, errors = prolog.solve(trace)
    record = {"index": index, "id": q["id"], "source": q.get("source_split"), "question": q["question"],
              "gold": q["answer"], "golds": ex.golds(q), "k": k, "system": SYSTEM, "kind": "prolog",
              "model": f"SWI-Prolog over the rules of {SOURCE}", "answer": answer, "votes": votes,
              "nars_answer": trace.get("answer"), "prolog_errors": errors, "error": None, "cost": 0.0,
              "started": datetime.now(timezone.utc).isoformat()}
    try:
        record.update(ex.metrics(judge, q, answer))
    except Exception as error:
        record["error"] = f"{type(error).__name__}: {error}"
    return record


for k in K_VALUES:
    done = ex.load_answers("results", SYSTEM, k)
    todo = [i for i in sorted(traces[k]) if i < len(questions) and i not in done]
    path = ex.answers_path("results", SYSTEM, k)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with ThreadPoolExecutor(8) as pool, open(path, "a") as f:
        for record in pool.map(lambda i: run_one(i, k), todo):
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    saved = ex.load_answers("results", SYSTEM, k)
    right = sum(r["judge"] for r in saved.values())
    print(f"✔ {SYSTEM} k={k:<2}: {right}/{len(saved)} judged correct ({100 * right / max(len(saved), 1):.1f}%) · "
          f"{len(todo)} run now")

## Prolog vs ONA on the same triples and rules

In [ ]:
from prettytable import PrettyTable

table = PrettyTable(["k", "questions", "ONA right", "Prolog right", "both", "only ONA", "only Prolog",
                     "ONA no answer", "Prolog no answer", "same answer"])
for k in K_VALUES:
    nars, pro = ex.load_answers("results", SOURCE, k), ex.load_answers("results", SYSTEM, k)
    common = sorted(nars.keys() & pro.keys())
    if not common:
        continue
    a = [nars[i]["judge"] for i in common]
    b = [pro[i]["judge"] for i in common]
    table.add_row([k, len(common), f"{100 * sum(a) / len(common):.1f}%", f"{100 * sum(b) / len(common):.1f}%",
                   sum(x and y for x, y in zip(a, b)), sum(x and not y for x, y in zip(a, b)),
                   sum(y and not x for x, y in zip(a, b)),
                   sum(not nars[i]["answer"] for i in common), sum(not pro[i]["answer"] for i in common),
                   sum(ex.normalize(nars[i]["answer"] or "") == ex.normalize(pro[i]["answer"] or "") for i in common)])
print(table)

## Report (every system saved in results/)

In [ ]:
ex.report("results", DATASET, N_QUESTIONS)